In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [14]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,Dropout,BatchNormalization,GlobalAveragePooling2D,Add,Activation,SeparableConv2D
from keras.models import Model
from tensorflow.keras.applications import MobileNetV2

from keras.layers import RandomFlip, RandomRotation, RandomZoom,RandomTranslation,RandomContrast, Input
from keras.callbacks import EarlyStopping,ModelCheckpoint

import sklearn
from sklearn.model_selection import train_test_split



In [15]:
df = pd.read_csv('./data/fer2013.csv')

In [16]:
df.head()

,emotion,pixels,Usage
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...,Training
1,0,151 150 147 155 148 133 111 140 170 174 182 15...,Training
2,2,231 212 156 164 174 138 161 173 182 200 106 38...,Training
3,4,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...,Training
4,6,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...,Training


In [17]:
#0: Angry
#1: Disgust
#2: Fear
#3: Happy
#4: Sad
#5: Surprise
#6: Neutral

In [18]:
df['emotion'].value_counts()

emotion
3    8989
6    6198
4    6077
2    5121
0    4953
5    4002
1     547
Name: count, dtype: int64

In [19]:
train_data = df[df['Usage'] == 'Training']
val_data = df[df['Usage'] == 'PublicTest']
test_data = df[df['Usage'] == 'PrivateTest']

def pixel_preprocess_rgb(df):
    x, y = [], []
    for _, row in df.iterrows():
        pixels = np.array(row['pixels'].split(), dtype='float32')
        pixels = pixels.reshape(48, 48, 1)
        pixels_rgb = np.repeat(pixels, 3, axis=-1)  # (48, 48, 3)
        x.append(pixels_rgb)
        y.append(row['emotion'])
    x = np.array(x) / 255.0
    y = keras.utils.to_categorical(np.array(y), num_classes=7)
    return x, y

X_train, y_train = pixel_preprocess_rgb(train_data)
X_val,   y_val   = pixel_preprocess_rgb(val_data)
X_test,  y_test  = pixel_preprocess_rgb(test_data)

print("Train",X_train.shape,y_train.shape)
print("Val",X_val.shape,y_val.shape)
print("Test",X_test.shape,y_test.shape)


KeyboardInterrupt: 

In [ ]:
X_train = X_train / 255.0
X_val   = X_val / 255.0
X_test  = X_test / 255.0

In [ ]:
data_aug = tf.keras.Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.06),
    RandomZoom(0.08),
    RandomTranslation(0.06, 0.06),
    RandomContrast(0.15)
])

In [ ]:

base_model = MobileNetV2(
    input_shape=(48, 48, 3),
    include_top=False,        
    weights='imagenet'       
)

base_model.trainable = False


inputs = Input(shape=(48, 48, 3))
x = data_aug(inputs, training=True)        
x = base_model(x, training=False)         
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
x = Dense(256, activation='relu')(x)      
x = Dropout(0.3)(x)
outputs = Dense(7, activation='softmax')(x)

model = Model(inputs, outputs)
model.summary()


C:\Users\thapa\AppData\Local\Temp\ipykernel_9012\3850704245.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 48, 48, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 48, 48, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 2, 2, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,587,719 (9.87 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

history_phase1 = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, lr_callback]
)

In [ ]:
# Unfreeze the top layers of the base model
base_model.trainable = True

# Only fine-tune the last 30 layers (earlier layers stay frozen)
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with a MUCH lower learning rate — critical
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),   # 100x smaller than phase 1
    loss=losss,
    metrics=['accuracy']
)

history_phase2 = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,                         # smaller batch helps fine-tuning
    validation_data=(X_val, y_val),
    callbacks=[early_stop, checkpoint, lr_callback]
)

In [20]:
losss = keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

checkpoint = ModelCheckpoint('emotion_model.h5',monitor='val_loss',save_best_only=True)

lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,    # reduce LR by half
    patience=3,    # if val_loss does not improve for 3 epochs
    verbose=1
)


In [31]:
optimizer = keras.optimizers.Adam(learning_rate=1e-3)

model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=['accuracy']
)

In [22]:
emotion_labels = [
    "Angry", 
    "Disgust", 
    "Fear", 
    "Happy", 
    "Sad", 
    "Surprise", 
    "Neutral"
]

In [1]:
import cv2
import numpy as np
import tensorflow
from tensorflow.keras.models import load_model

# Load trained model
model = load_model("emotion_model.h5")

emotion_labels = [
    "Angry", 
    "Disgust", 
    "Fear", 
    "Happy", 
    "Sad", 
    "Surprise", 
    "Neutral"
]

# Load face detector (Haar cascade)
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]

        # Resize to 48x48 (same as training)
        face = cv2.resize(face, (48, 48))

        # Normalize like training
        face = face / 255.0

        # Reshape to model input
        face = np.reshape(face, (1, 48, 48, 1))

        # Predict
        predictions = model.predict(face, verbose=0)
        emotion = emotion_labels[np.argmax(predictions)]

        # Draw rectangle + label
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, emotion, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.9, (0,255,0), 2)

    cv2.imshow("Emotion Detector", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
